<!-- track-identity-card -->
# Robustness and null calibration audit

| | |
|---|---|
| Pipeline step | `09_robustness.ipynb` |
| Manuscript section | 4.2, 4.3 |
| Copied from | `notebooks/NB17_robustness_v1_2026-07-23_1500.ipynb` |
| Source sha256 | `21dabf143000c2f7f8bba911f2f03a09` |

**Reads**

- `data/fingerprints/**`
- `results/cross_car_<policy>/nb16_info.json`

**Writes**

- `results/robustness/nb17_robustness_<stamp>.json`

Audit layer. Recomputes the null distributions behind the Section 4.3 table, and the agreement between the within-type clusterings reported in Section 4.2.

> Copied verbatim from the working notebook. The identity card above is the only addition; no code cell was modified.


# NB17 - Saglamlik Analizleri (v1)

Mevcut ciktilari **okur**, hicbir seyi yeniden uretmez. Tek kosuda iki politika.

| Kod | Analiz | Hedef |
|-----|--------|-------|
| B3 | Algoritma uyumu (KMeans / Agglomerative / FCM) | K3 - sifir maliyet, veri hazir |
| B2 | k duyarliligi (k = 2, 3, 4) | K3 - "neden k=3" sorusuna cevap |
| B1 | Permutasyon null (silhouette, ARI) | K1 + K3 - mutlak referans |
| A1 | Bootstrap guven araliklari | K1 + K3 - nokta tahmini yerine aralik |
| A2 | Kisi duzeyi grupli capraz dogrulama | K3 - kardes kayit sizintisini kapatir |
| B4 | Gereken N (Bonett) | K2 - "gucsuzduk" yerine tasarim ifadesi |

**Onemli:** A2 kayit duzeyi LOOCV'yi de yeniden hesaplar ve saklanan
`loocv_accuracy` ile karsilastirir. Eslesme, yeniden kurulan ozellik
matrisinin dogru oldugunun kanitidir; eslesmezse grupli CV sayisi da
guvenilmez demektir.

In [ ]:
# track-config-bootstrap
# Locates track/config.py, which resolves the data root at run time.
# Works from a flat layout (track/ beside the notebooks) and from the
# repository layout (src/track/ one level up). See track/config.py.
import sys as _sys, pathlib as _pl
_cands = []
for _p in [_pl.Path.cwd()] + list(_pl.Path.cwd().parents):
    _cands += [_p, _p / "src"]
for _c in _cands:
    if (_c / "track" / "config.py").is_file():
        _sys.path.insert(0, str(_c))
        break
else:
    raise RuntimeError(
        "track/config.py not found. Run this notebook from inside the repository, "
        "or add the directory holding track/ to sys.path."
    )
from track.config import PROJECT_ROOT as TRACK_ROOT
print("data root:", TRACK_ROOT)


In [ ]:
# %% 1. YAPILANDIRMA
from pathlib import Path
import warnings, json, itertools
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import (silhouette_score, adjusted_rand_score, accuracy_score)
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneOut, LeaveOneGroupOut, cross_val_predict

warnings.filterwarnings("ignore")

PROJECT   = TRACK_ROOT
FP_DIR    = PROJECT / "data" / "fingerprints"
FEAT_DIR  = PROJECT / "data" / "features"
RES_DIR   = PROJECT / "results"
OUT_DIR   = RES_DIR / "robustness"
OUT_DIR.mkdir(parents=True, exist_ok=True)

POLICIES = {"P_INC": "_inc", "P_EXC": "_exc"}      # ikisi de tek kosuda
TIERS    = ["T1", "T2", "T3", "T4"]
TRACKS   = ["monza", "barcelona", "red_bull_ring"]
CORNER_ORDER = ["slow", "medium", "fast"]

SEED      = 42
N_BOOT    = 1000        # bootstrap tekrari
N_PERM    = 1000        # permutasyon tekrari
SUBSAMPLE = 0.80        # bootstrap: m-out-of-n ORNEKLEMSIZ
K_GRID    = [2, 3, 4]
K_MAIN    = 3           # NB11 ile ayni: kademeler arasi karsilastirilabilirlik icin sabit

DIMENSIONS = {
    "B1_Hiz":        ["mean_apex_speed", "speed_loss_eff", "mean_mc_speed_ratio",
                      "mean_mc_lateral", "mean_cex_accel_rate"],
    "B2_Frenleme":   ["mean_brake_pressure", "trail_braking_ratio", "mean_trail_pressure",
                      "mean_slb_dist", "mean_slb_decel", "mean_ce_brake_turnin"],
    "B3_Strateji":   ["mean_coasting_dist", "mean_cex_throttle_lag",
                      "pct_lift_coast", "pct_flat_out"],
    "B4_Tutarlilik": ["apex_speed_std", "exit_speed_std",
                      "speed_loss_eff_std", "braking_dist_std"],
}
DIM_NAMES   = list(DIMENSIONS)
ALL_METRICS = [m for v in DIMENSIONS.values() for m in v]

CT_METRICS = ["mean_apex_speed", "std_apex_speed", "mean_braking_dist",
              "std_braking_dist", "mean_exit_speed", "std_exit_speed", "trail_pct"]

RESULTS = {}
print("Proje :", PROJECT, "| var mi:", PROJECT.exists())
print("Cikti :", OUT_DIR)
print(f"Ayar  : boot={N_BOOT} perm={N_PERM} altornek={SUBSAMPLE} k={K_GRID} seed={SEED}")

In [ ]:
# %% 2. YARDIMCILAR
def person_of(driver_id):
    """`20240215_154341_HC` -> `HC` ; `20240410_A_12_123` -> `A_12_123`.
    Bastaki tum SAYISAL parcalari atar (tarih + varsa saat)."""
    parts = str(driver_id).split("_")
    i = 0
    while i < len(parts) - 1 and parts[i].isdigit():
        i += 1
    return "_".join(parts[i:])


def load_tier(tier, suf):
    """Kademe parmak izini ve ham metrik matrisini yukler."""
    fp_path = FP_DIR / (tier.lower() + suf) / "fingerprint_cross_track.parquet"
    if not fp_path.exists():
        return None, None, None
    fp = pd.read_parquet(fp_path)
    info_path = FP_DIR / (tier.lower() + suf) / f"{tier.lower()}_info.json"
    info = json.load(open(info_path, encoding="utf-8")) if info_path.exists() else {}

    # X_raw'i NB11 ile AYNI sekilde yeniden kur: pistler arasi ortalama
    feat = FEAT_DIR / (tier.lower() + suf)
    mats = {}
    for t in TRACKS:
        p = feat / f"driver_corner_matrix_{t}.parquet"
        if p.exists():
            mats[t] = pd.read_parquet(p)
    if not mats:
        return fp, None, info
    avail = [m for m in ALL_METRICS if any(m in mm.columns for mm in mats.values())]
    rows = []
    for did in fp["driver_id"]:
        row = {"driver_id": did}
        for m in avail:
            vals = [float(mm[mm["driver_id"] == did].iloc[0][m])
                    for mm in mats.values()
                    if m in mm.columns and (mm["driver_id"] == did).any()
                    and pd.notna(mm[mm["driver_id"] == did].iloc[0][m])]
            row[m] = np.mean(vals) if vals else np.nan
        rows.append(row)
    raw = pd.DataFrame(rows)
    return fp, raw[["driver_id"] + avail], info


def sil_at_k(X, k, seed=SEED):
    if len(X) <= k:
        return np.nan
    lab = KMeans(n_clusters=k, random_state=seed, n_init=20).fit_predict(X)
    if len(set(lab)) < 2:
        return np.nan
    return float(silhouette_score(X, lab))


def boot_ci(stat_fn, n_items, n_boot=N_BOOT, frac=SUBSAMPLE, seed=SEED):
    """m-out-of-n ORNEKLEMSIZ alt-orneklem. Yerine-koymali bootstrap
    kume gecerlilik olculerini sisirir (ayni nokta -> sifir uzaklik)."""
    rng = np.random.default_rng(seed)
    m = max(4, int(np.ceil(frac * n_items)))
    out = []
    for _ in range(n_boot):
        idx = rng.choice(n_items, size=m, replace=False)
        v = stat_fn(idx)
        if v is not None and np.isfinite(v):
            out.append(v)
    if len(out) < 20:
        return dict(n_ok=len(out), lo=None, med=None, hi=None)
    a = np.array(out)
    return dict(n_ok=len(a), lo=float(np.percentile(a, 2.5)),
                med=float(np.median(a)), hi=float(np.percentile(a, 97.5)))


def perm_null_columns(X, k, n_perm=N_PERM, seed=SEED):
    """Null: her ozellik sutunu BAGIMSIZ karistirilir. Marjinaller korunur,
    ozellikler arasi yapi (yani kume yapisi) yok edilir."""
    rng = np.random.default_rng(seed)
    vals = []
    for _ in range(n_perm):
        Xp = np.column_stack([rng.permutation(X[:, j]) for j in range(X.shape[1])])
        v = sil_at_k(Xp, k)
        if np.isfinite(v):
            vals.append(v)
    a = np.array(vals)
    return a


def p_from_null(obs, null_arr):
    """Tek yonlu (buyuk = daha yapisal) permutasyon p, +1 duzeltmeli."""
    if len(null_arr) == 0 or not np.isfinite(obs):
        return None
    return float((np.sum(null_arr >= obs) + 1) / (len(null_arr) + 1))


# --- sentetik dogrulama: fonksiyonlar beklendigi gibi mi ---
_rng = np.random.default_rng(0)
_X_yapili = np.vstack([_rng.normal(m, .35, (12, 4)) for m in (0, 3, 6)])
_X_gurultu = _rng.normal(0, 1, (36, 4))
_s1, _s0 = sil_at_k(_X_yapili, 3), sil_at_k(_X_gurultu, 3)
assert _s1 > _s0, "sil_at_k yapili veriyi ayirt edemiyor"
assert person_of("20240215_154341_HC") == "HC"
assert person_of("20240426_NiM") == "NiM"
assert person_of("20240410_A_12_123") == "A_12_123"
print(f"Sentetik denetim OK  (yapili sil={_s1:.3f} > gurultu sil={_s0:.3f})")

## B3 - Algoritma uyumu

Sifir maliyet: uc kumeleme algoritmasinin silhouette degerleri her kademede
zaten hesaplanmis, JSON'da duruyor, hicbir yerde raporlanmiyor. Ucu de ayni
yone isaret ediyorsa sonuc **algoritma secimine bagli degil** demektir.

In [ ]:
# %% 3. B3 - ALGORITMA UYUMU (sifir maliyet)
rows = []
for pol, suf in POLICIES.items():
    for tier in TIERS:
        p = FP_DIR / (tier.lower() + suf) / f"{tier.lower()}_info.json"
        if not p.exists():
            print("  eksik:", p); continue
        d = json.load(open(p, encoding="utf-8"))
        rows.append(dict(politika=pol, kademe=tier,
                         kmeans=d.get("silhouette_k3"),
                         agglomerative=d.get("silhouette_agglomerative_k3"),
                         fcm=d.get("silhouette_fcm_k3"),
                         en_iyi_k=d.get("silhouette_best_k"),
                         n_kayit=d.get("n_drivers_cross_track")))
b3 = pd.DataFrame(rows)
if len(b3):
    b3["yayilim"] = (b3[["kmeans", "agglomerative", "fcm"]].max(axis=1)
                     - b3[["kmeans", "agglomerative", "fcm"]].min(axis=1))
    print(b3.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
    print("\n--- Monotonluk (T1 -> T4), algoritma basina ---")
    for pol in POLICIES:
        s = b3[b3.politika == pol].set_index("kademe")
        for alg in ["kmeans", "agglomerative", "fcm"]:
            v = s.loc[[t for t in TIERS if t in s.index], alg].astype(float).values
            azalan = bool(np.all(np.diff(v[:3]) < 0))   # T1->T2->T3 (T4=T3)
            print(f"  {pol} {alg:<14} {np.round(v,4)}  T1>T2>T3 monoton azalan: {azalan}")
    RESULTS["B3_algoritma_uyumu"] = b3.to_dict("records")
else:
    print("B3 atlandi - JSON bulunamadi")

## B2 - k duyarliligi

NB11 kademeler arasi karsilastirilabilirlik icin k=3'u sabit tutuyor
(`# k=3 ile kumeleme (T1 ile karsilastirma icin)`). Bu savunulabilir bir
tasarim karari, ama kanit gerektiriyor: **monotonluk k=2 ve k=4'te de
suruyor mu?** Suruyorsa sonuc k secimine bagli degildir.

In [ ]:
# %% 4. B2 - k DUYARLILIGI
rows, TIER_CACHE = [], {}
for pol, suf in POLICIES.items():
    for tier in TIERS:
        fp, raw, info = load_tier(tier, suf)
        if fp is None:
            print("  eksik:", tier, pol); continue
        TIER_CACHE[(pol, tier)] = (fp, raw, info)
        X = fp[DIM_NAMES].fillna(0.5).values
        r = dict(politika=pol, kademe=tier, n=len(fp))
        for k in K_GRID:
            r[f"k{k}"] = sil_at_k(X, k)
        rows.append(r)
b2 = pd.DataFrame(rows)
if len(b2):
    print(b2.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
    print("\n--- T1 -> T3 monotonluk, k basina ---")
    for pol in POLICIES:
        s = b2[b2.politika == pol].set_index("kademe")
        for k in K_GRID:
            v = s.loc[[t for t in ["T1", "T2", "T3"] if t in s.index], f"k{k}"].astype(float).values
            print(f"  {pol} k={k}: {np.round(v,4)}  monoton azalan: {bool(np.all(np.diff(v) < 0))}")
    RESULTS["B2_k_duyarliligi"] = b2.to_dict("records")

## B1 + A1 - Silhouette icin permutasyon null'i ve bootstrap araligi

Silhouette 0.34 iyi mi kotu mu? Su an referans noktasi yok. Iki olcum:

* **Permutasyon null'i** - her ozellik sutunu bagimsiz karistirilir; marjinal
  dagilimlar korunur, kume yapisi yok edilir. Gozlenen deger null'in neresinde?
* **Bootstrap araligi** - %80 alt-orneklem (orneklemsiz). Yerine-koymali
  bootstrap kume olculerini sisirir; ayni nokta iki kez cekilince aralarindaki
  uzaklik sifir olur.

In [ ]:
# %% 5. B1 + A1 - PERMUTASYON NULL + BOOTSTRAP
rows = []
for (pol, tier), (fp, raw, info) in sorted(TIER_CACHE.items()):
    X = fp[DIM_NAMES].fillna(0.5).values
    obs = sil_at_k(X, K_MAIN)
    null = perm_null_columns(X, K_MAIN)
    ci = boot_ci(lambda idx: sil_at_k(X[idx], K_MAIN), len(X))
    rows.append(dict(politika=pol, kademe=tier, n=len(X), gozlenen=obs,
                     null_ort=float(np.mean(null)) if len(null) else None,
                     null_p95=float(np.percentile(null, 95)) if len(null) else None,
                     p_perm=p_from_null(obs, null),
                     boot_lo=ci["lo"], boot_med=ci["med"], boot_hi=ci["hi"]))
    _nu = f"{np.mean(null):.4f}" if len(null) else "-"
    _ga = (f"[{ci['lo']:.3f}, {ci['hi']:.3f}]" if ci["lo"] is not None else "-")
    print(f"  {pol} {tier}: gozlenen={obs:.4f}  null_ort={_nu}  "
          f"p={p_from_null(obs, null)}  GA={_ga}")
b1 = pd.DataFrame(rows)
if len(b1):
    print()
    print(b1.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
    print("\nYORUM: p_perm kucukse gozlenen yapi rastgeleden ayirt edilebilir.")
    print("       Kademelerin GA'lari ORTUSUYORSA dususun 'kesin' oldugu soylenemez.")
    RESULTS["B1_A1_silhouette"] = b1.to_dict("records")

## A2 - Kisi duzeyi grupli capraz dogrulama

NB11'in LOOCV'si **kayit** duzeyinde: ayni kisinin birden fazla oturum-gunu
varsa (T2-T4'te HC uc kez) biri disarida birakilirken kardesleri egitimde
kaliyor. Model kisiyi kardesten okuyabilir.

Asagida ikisi de hesaplanir. **Kayit duzeyi sayi saklanan `loocv_accuracy`
ile eslesmeli** - eslesme, ozellik matrisinin dogru yeniden kuruldugunun
kanitidir. Eslesmezse grupli sayi da guvenilmez.

In [ ]:
# %% 6. A2 - KISI DUZEYI GRUPLI CV
rows = []
for (pol, tier), (fp, raw, info) in sorted(TIER_CACHE.items()):
    if raw is None or len(fp) < 6:
        print(f"  {pol} {tier}: atlandi (ham matris yok ya da n<6)"); continue
    X = fp[DIM_NAMES].fillna(0.5).values
    y = KMeans(n_clusters=K_MAIN, random_state=SEED, n_init=20).fit_predict(X)

    metrics = [c for c in raw.columns if c != "driver_id"]
    Xr = raw[metrics].fillna(0).values
    persons = np.array([person_of(d) for d in fp["driver_id"]])

    def _rf():
        return RandomForestClassifier(n_estimators=500, max_depth=4,
                                      min_samples_leaf=3, random_state=SEED,
                                      class_weight="balanced")

    acc_rec = accuracy_score(y, cross_val_predict(_rf(), Xr, y, cv=LeaveOneOut()))
    n_gr = len(set(persons))
    if n_gr >= 3:
        acc_grp = accuracy_score(y, cross_val_predict(_rf(), Xr, y,
                                                      cv=LeaveOneGroupOut(), groups=persons))
    else:
        acc_grp = np.nan
    saklanan = info.get("loocv_accuracy")
    eslesti = (saklanan is not None) and abs(acc_rec - float(saklanan)) < 1e-9
    rows.append(dict(politika=pol, kademe=tier, n_kayit=len(y), n_kisi=n_gr,
                     kayit_loocv=acc_rec, saklanan_loocv=saklanan, eslesti=eslesti,
                     kisi_grupli_cv=acc_grp,
                     fark=(acc_grp - acc_rec) if np.isfinite(acc_grp) else None))
    bay = "OK" if eslesti else "!! ESLESMEDI"
    print(f"  {pol} {tier}: kayit={acc_rec:.4f} (saklanan={saklanan}) {bay} | "
          f"grupli={acc_grp:.4f} | kayit={len(y)} kisi={n_gr}")
a2 = pd.DataFrame(rows)
if len(a2):
    print()
    print(a2.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
    n_es = int(a2["eslesti"].sum())
    print(f"\nYENIDEN KURULUM DENETIMI: {n_es}/{len(a2)} kademe saklanan degerle eslesti.")
    if n_es < len(a2):
        print("  UYARI: eslesmeyen kademelerde grupli CV sayisi KULLANILMAMALI.")
    RESULTS["A2_grupli_cv"] = a2.to_dict("records")

## B1 + A1 (K1) - ARI icin bootstrap araligi ve permutasyon null'i

ARI iki politikada 0.043 ve 0.271 - alti kat fark. Nokta tahmini olarak bu
"politikaya asiri duyarli" gorunur. **Guven araliklari ortusuyorsa** dogru
yorum farkli: her ikisi de ayni belirsiz buyuklugun gurultulu kestirimleri.
Bu, mevcut anlatidan hem daha dogru hem daha savunulabilir.

ARI zaten sansa gore duzeltilmis (beklenen deger ~0), ama null'in **yayilimi**
n=21-24'te dar mi genis mi bilinmiyor. Permutasyon onu olcer.

In [ ]:
# %% 7. K1 - ARI BOOTSTRAP + PERMUTASYON
def per_ct_labels_from(df_cross, k=K_MAIN, seed=SEED):
    """Viraj tipi basina KMeans - NB7A7b ile ayni: MinMax + k=3 + n_init=20."""
    out = {}
    for ct in CORNER_ORDER:
        d = df_cross[df_cross["corner_type"] == ct].set_index("driver_id")
        cols = [c for c in CT_METRICS if c in d.columns]
        if len(d) <= k or not cols:
            continue
        Xs = MinMaxScaler().fit_transform(d[cols].values)
        out[ct] = pd.Series(KMeans(n_clusters=k, random_state=seed,
                                   n_init=20).fit_predict(Xs), index=d.index)
    return out


def mean_offdiag_ari(labels):
    vals = []
    for a, b in itertools.combinations(CORNER_ORDER, 2):
        if a in labels and b in labels:
            common = labels[a].index.intersection(labels[b].index)
            if len(common) >= 4:
                vals.append(adjusted_rand_score(labels[a].loc[common].values,
                                                labels[b].loc[common].values))
    return float(np.mean(vals)) if vals else np.nan


rows = []
for pol, suf in POLICIES.items():
    p = FP_DIR / ("corner_type_profiles" + suf) / "corner_type_profile_crosstrack.parquet"
    if not p.exists():
        print("  eksik:", p); continue
    dfc = pd.read_parquet(p)
    drivers = sorted(dfc["driver_id"].unique())
    obs = mean_offdiag_ari(per_ct_labels_from(dfc))

    def _stat(idx):
        keep = [drivers[i] for i in idx]
        return mean_offdiag_ari(per_ct_labels_from(dfc[dfc["driver_id"].isin(keep)]))

    ci = boot_ci(_stat, len(drivers), n_boot=max(200, N_BOOT // 5))

    rng = np.random.default_rng(SEED)
    base = per_ct_labels_from(dfc)
    null = []
    for _ in range(N_PERM):
        sh = {ct: pd.Series(rng.permutation(s.values), index=s.index)
              for ct, s in base.items()}
        v = mean_offdiag_ari(sh)
        if np.isfinite(v):
            null.append(v)
    null = np.array(null)

    rows.append(dict(politika=pol, n_surucu=len(drivers), gozlenen_ari=obs,
                     boot_lo=ci["lo"], boot_med=ci["med"], boot_hi=ci["hi"],
                     null_ort=float(np.mean(null)) if len(null) else None,
                     null_p95=float(np.percentile(null, 95)) if len(null) else None,
                     p_perm=p_from_null(obs, null)))
    print(f"  {pol}: ARI={obs:.4f}  GA=[{ci['lo']}, {ci['hi']}]  "
          f"null_ort={np.mean(null):.4f}  p={p_from_null(obs, null)}")

k1 = pd.DataFrame(rows)
if len(k1):
    print()
    print(k1.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
    if len(k1) == 2 and all(k1["boot_lo"].notna()):
        a, b = k1.iloc[0], k1.iloc[1]
        ortusuyor = (a["boot_lo"] <= b["boot_hi"]) and (b["boot_lo"] <= a["boot_hi"])
        print(f"\nIKI POLITIKANIN GA'LARI ORTUSUYOR MU: {ortusuyor}")
        print("  Ortusuyorsa: 'politikaya asiri duyarli' yerine 'her ikisi de ayni")
        print("  belirsiz buyuklugun gurultulu kestirimi' yazilmali.")
    RESULTS["K1_ari"] = k1.to_dict("records")

## B4 - Gereken N (Bonett yaklasimi)

"Gucsuzduk" bir itiraf; "bu etkiyi su kesinlikle kestirmek icin N kisi
gerekirdi" bir tasarim ifadesidir. Hakemin okumak istedigi ikincisidir.

Asagidaki hucre once formulu saklanan `apriori_precision_bonett` degerleriyle
**dogrular**; tutuyorsa ayni formulu tersine cevirip gereken N'i verir.

In [ ]:
# %% 8. B4 - GEREKEN N
Z, K_MEAS = 1.96, 3      # K_MEAS=3: NB16'nin sakladigi degerleri ureten varsayim

def bonett_genislik(R, n, k=K_MEAS, z=Z):
    return float(np.sqrt(8 * z**2 * (1 - R)**2 * (1 + (k - 1) * R)**2 /
                         (k * (k - 1) * (n - 1))))

def bonett_n(R, w, k=K_MEAS, z=Z):
    return int(np.ceil(8 * z**2 * (1 - R)**2 * (1 + (k - 1) * R)**2 /
                       (k * (k - 1) * w**2) + 1))

for pol, suf in POLICIES.items():
    p = RES_DIR / f"cross_car{suf}" / "nb16_info.json"
    if not p.exists():
        print("  eksik:", p); continue
    d = json.load(open(p, encoding="utf-8"))
    n_kisi = int(d.get("ladder", {}).get("n_persons", 0)) or None
    saklanan = d.get("apriori_precision_bonett", {})
    print(f"\n=== {pol}  (n = {n_kisi} kisi) ===")
    print("  --- formul denetimi: saklanan vs yeniden hesaplanan GA genisligi ---")
    _farklar = []
    for Rs, w_sak in sorted(saklanan.items(), key=lambda kv: float(kv[0])):
        R = float(Rs)
        w_hes = bonett_genislik(R, n_kisi) if n_kisi else np.nan
        _farklar.append(abs(float(w_sak) - w_hes))
        print(f"    R={R:.1f}  saklanan={float(w_sak):.3f}  hesaplanan={w_hes:.3f}"
              f"   fark={abs(float(w_sak)-w_hes):.3f}")
    if _farklar:
        _mf = float(np.mean(_farklar))
        _ok = _mf < 0.05
        print(f"    ortalama mutlak fark = {_mf:.4f}  ->  "
              + ("formul dogrulandi (Bonett'in yaklasik varyantlari arasindaki "
                 "kucuk fark)" if _ok else
                 "!! FORMUL TUTMUYOR - asagidaki gereken-N sayilari KULLANILMAMALI"))
    print("  --- gereken N (hedef GA genisligi basina) ---")
    print(f"    {'R':>5} {'w=0.40':>8} {'w=0.30':>8} {'w=0.20':>8}")
    for R in [0.3, 0.4, 0.5, 0.6, 0.7]:
        print(f"    {R:>5.1f} " + " ".join(f"{bonett_n(R, w):>8d}" for w in (0.40, 0.30, 0.20)))

    # gozlenen en guclu metrik icin somut cumle
    rep = sorted([r for r in d.get("repeatability", []) if r.get("fdr_anlamli")],
                 key=lambda r: -r["R"])
    if rep:
        r0 = rep[0]
        print(f"\n  En yuksek tekrarlanabilirlik: {r0['metrik']} R={r0['R']:.3f} "
              f"[{r0['ci_lo']:.3f}, {r0['ci_hi']:.3f}] (genislik {r0['ci_hi']-r0['ci_lo']:.3f})")
        print(f"  Ayni R'yi w=0.20 kesinlikle kestirmek icin gereken N: "
              f"{bonett_n(r0['R'], 0.20)} kisi")
    RESULTS.setdefault("B4_gereken_n", {})[pol] = dict(
        n_kisi=n_kisi, k_olcum=K_MEAS,
        gereken_n={f"R{R}_w{w}": bonett_n(R, w)
                   for R in (0.3, 0.4, 0.5, 0.6, 0.7) for w in (0.40, 0.30, 0.20)})

In [ ]:
# %% 9. IZLENEBILIRLIK + OZET
from datetime import datetime as _dt
RESULTS["_meta"] = dict(
    notebook="NB17_robustness_v1",
    run_timestamp=_dt.now().isoformat(timespec="seconds"),
    seed=SEED, n_boot=N_BOOT, n_perm=N_PERM, subsample=SUBSAMPLE,
    k_grid=K_GRID, k_main=K_MAIN,
    bootstrap_yontemi="m-out-of-n ORNEKLEMSIZ alt-orneklem (%80); "
                      "yerine-koymali bootstrap kume gecerlilik olculerini sisirir",
    perm_null_yontemi="her ozellik sutunu bagimsiz karistirilir "
                      "(marjinaller korunur, kume yapisi yok edilir)",
    politikalar=list(POLICIES), kademeler=TIERS,
    not_="Salt-okunur. Hicbir birincil sonuc yeniden uretilmedi.")

out = OUT_DIR / f"nb17_robustness_{_dt.now():%Y-%m-%d_%H%M}.json"
with open(out, "w", encoding="utf-8") as f:
    json.dump(RESULTS, f, indent=2, ensure_ascii=False, default=str)
print("Yazildi:", out)

print("\n" + "=" * 62)
print("OZET - metne girecek uc soru")
print("=" * 62)
if "B3_algoritma_uyumu" in RESULTS:
    y = pd.DataFrame(RESULTS["B3_algoritma_uyumu"])["yayilim"].max()
    print(f"1. Algoritma secimine bagli mi?   en buyuk yayilim = {y:.4f}")
if "A2_grupli_cv" in RESULTS:
    a = pd.DataFrame(RESULTS["A2_grupli_cv"])
    print(f"2. LOOCV sizintili miydi?         yeniden kurulum {int(a['eslesti'].sum())}/{len(a)}; "
          f"ortalama fark = {a['fark'].astype(float).mean():+.4f}")
if "K1_ari" in RESULTS:
    k = pd.DataFrame(RESULTS["K1_ari"])
    print(f"3. ARI gercekten duyarli mi?      GA'lar: " +
          " | ".join(f"{r.politika} [{r.boot_lo:.3f}, {r.boot_hi:.3f}]"
                     for r in k.itertuples() if pd.notna(r.boot_lo)))

In [ ]:
# ==============================================================
# D-LISTESI  ·  §4.4 ve §4.2'nin dogrulanmamis sayilari
# SALT OKUNUR. Hicbir dosya yazilmaz.
# ==============================================================
import json
from pathlib import Path

ROOT = TRACK_ROOT

jsonlar = [p for p in ROOT.rglob("nb16_info.json")
           if p.parent.parent.name.lower() == "results"
           and ("cross_car_exc" in str(p) or "cross_car_inc" in str(p))
           and "_arsiv" not in str(p).lower()]

for jp in sorted(jsonlar):
    J = json.load(open(jp, encoding="utf-8"))
    print("=" * 72)
    print("%s   (%s)" % (J["identity_policy"], jp.parent.name))
    print("=" * 72)

    # ---- D1 . pist-ici karsilastirmalar: kaci n>=3, kaci anlamli
    wt = J.get("identifiability_within_track") or []
    gecerli = [r for r in wt if (r.get("n") or 0) >= 3]
    print("D1 . PIST-ICI  (metin 'seven' diyor)")
    print("     toplam satir %d · n>=3 olan %d" % (len(wt), len(gecerli)))
    for r in wt:
        print("     %-14s %-14s x %-14s n=%-3s acc=%-7s p=%-8s %s"
              % (r.get("track"), r.get("car_a"), r.get("car_b"), r.get("n"),
                 round(r["accuracy"], 4) if r.get("accuracy") is not None else "--",
                 round(r["p_perm"], 4) if r.get("p_perm") is not None else "--",
                 "ANLAMLI" if r.get("anlamli") else ""))
    print("     -> n>=3 icinde anlamli: %d"
          % sum(1 for r in gecerli if r.get("anlamli")))

    # ---- D2 . s2_derived tam dokum (aralik 0.18-0.47 iddiasi)
    print("\nD2 . S2 TURETILMIS ORANLAR  (metin '0.18 to 0.47' diyor)")
    for r in (J.get("s2_derived") or []):
        print("     " + json.dumps(r, ensure_ascii=False))
    for r in (J.get("s2_dropped") or []):
        print("     DUSEN " + json.dumps(r, ensure_ascii=False))

    # ---- D3 . Bonett beklenen genislik (metin '0.62' diyor)
    print("\nD3 . BONETT BEKLENEN GA GENISLIGI  (metin '0.62' diyor)")
    ap = J.get("apriori_precision_bonett") or {}
    for k in sorted(ap, key=float):
        print("     R=%-5s -> genislik %.4f" % (k, ap[k]))

    # ---- yan: manset metrigin gozlenen genisligi
    rep = {r["metrik"]: r for r in J.get("repeatability", [])}
    mas = rep.get("mean_apex_speed")
    if mas:
        print("     gozlenen: R=%.4f · genislik %.4f" % (mas["R"], mas["ci_genislik"]))
    print()

# ---- D4 . §4.2 ARI grup-cikarma: p5 / p95 nerede
print("=" * 72)
print("D4 . ARI GRUP-CIKARMA  (metin p5=0.109 / p95=0.294 diyor)")
print("=" * 72)
adaylar = [p for p in ROOT.rglob("*.json")
           if "_arsiv" not in str(p).lower() and "backup" not in str(p).lower()
           and p.stat().st_size < 20_000_000]
bulundu = 0
for p in adaylar:
    try:
        metin = open(p, encoding="utf-8", errors="ignore").read()
    except Exception:
        continue
    if "0.109" in metin and "0.294" in metin:
        print("   ADAY: %s" % p)
        bulundu += 1
        try:
            J = json.load(open(p, encoding="utf-8"))
            def gez(o, pre=""):
                if isinstance(o, dict):
                    for k, v in o.items():
                        yol = ("%s.%s" % (pre, k)) if pre else k
                        if isinstance(v, (dict, list)):
                            gez(v, yol)
                        elif any(s in str(k).lower()
                                 for s in ("p5", "p95", "pct", "percentile",
                                           "yuzdelik", "sd", "std", "mean", "ort")):
                            print("      %-52s %s" % (yol, v))
                elif isinstance(o, list):
                    for i, v in enumerate(o[:6]):
                        if isinstance(v, (dict, list)):
                            gez(v, "%s[%d]" % (pre, i))
            gez(J)
        except Exception as e:
            print("      (JSON okunamadi: %s)" % type(e).__name__)
if not bulundu:
    print("   0.109 ve 0.294'u BIRLIKTE tasiyan JSON YOK.")
    print("   -> sayilar baska bir ciktidan (parquet/csv/defter hucresi) gelmis olabilir;")
    print("      kaynak bulunana kadar [DOGRULANMADI] kalir.")

print("\nBITTI -- hicbir dosya yazilmadi.")